# UNCP 6 | Model and executed results

کد مدل و یک بررسی اجرایی در همین دفترچه اجرا شده‌اند. اعداد سه seed از فایل‌های نتیجهٔ آزمایش کامل که پیش‌تر اجرا شده‌اند، در سلول آخر ثبت و با میانگین همان فایل‌ها کنترل می‌شوند. این دفترچه آموزش کامل سه seed را دوباره اجرا نمی‌کند؛ برای بازآموزی، داده، کد benchmark و تنظیمات همان پروژه لازم است.

In [1]:
import torch
from torch import nn
import torch.nn.functional as F

"""UNCP 6: repeat-aware dual-scale click model.

One GRU reads every observed click; a second GRU reads consecutive runs with
their lengths. A repeat-vs-switch hazard and a conditional switch classifier
form a normalized next-click distribution. No target information is input.
"""
import torch
from torch import nn
import torch.nn.functional as F


class UNCP6NextClick(nn.Module):
    def __init__(self, vocab_size, n_classes, config):
        super().__init__()
        dim, hidden = config['embedding_dim'], config['hidden_dim']
        self.embedding = nn.Embedding(vocab_size, dim, padding_idx=0)
        self.run_length_embedding = nn.Embedding(config['max_context'] + 1, dim)
        self.raw_gru = nn.GRU(dim, hidden, batch_first=True)
        self.run_gru = nn.GRU(dim, hidden, batch_first=True)
        self.gate = nn.Linear(2 * hidden, hidden)
        self.norm = nn.LayerNorm(hidden)
        self.dropout = nn.Dropout(config['dropout'])
        self.repeat_head = nn.Linear(hidden, 1)
        self.switch_head = nn.Linear(hidden, n_classes)
        self.register_buffer('repeat_class', config['repeat_class'].long())

    def forward(self, x, lengths):
        x = x[:, :int(lengths.max())]
        batch, steps = x.shape
        rows = torch.arange(batch, device=x.device)
        positions = torch.arange(steps, device=x.device)[None, :]
        valid = positions < lengths[:, None]
        keep = valid.clone()
        keep[:, 1:] &= x[:, 1:].ne(x[:, :-1])
        destination = keep.long().cumsum(1) - 1
        compact = torch.zeros_like(x)
        compact[rows[:, None].expand_as(x)[keep], destination[keep]] = x[keep]
        run_lengths = torch.zeros_like(x)
        run_lengths.scatter_add_(1, destination.clamp_min(0), valid.long())
        compact_lengths = keep.sum(1)
        raw, _ = self.raw_gru(self.embedding(x))
        raw = raw[rows, lengths - 1]
        run_inputs = self.embedding(compact) + self.run_length_embedding(run_lengths.clamp_max(self.run_length_embedding.num_embeddings - 1))
        runs, _ = self.run_gru(run_inputs)
        runs = runs[rows, compact_lengths - 1]
        gate = torch.sigmoid(self.gate(torch.cat([raw, runs], dim=-1)))
        state = self.dropout(self.norm(gate * raw + (1 - gate) * runs))
        repeat_logit = self.repeat_head(state).squeeze(-1)
        switch_logits = self.switch_head(state)
        last = x[rows, lengths - 1]
        repeat_index = self.repeat_class[last]
        has_repeat_class = repeat_index.ge(0)
        active_repeat_index = repeat_index.clamp_min(0)
        mask = F.one_hot(active_repeat_index, switch_logits.shape[-1]).bool() & has_repeat_class[:, None]
        switch_logits = switch_logits.masked_fill(mask, -1e9)
        switch_log_probs = F.log_softmax(switch_logits, dim=-1)
        repeat_log_prob = F.logsigmoid(repeat_logit)
        switch_log_prob = F.logsigmoid(-repeat_logit)
        # Unknown input labels cannot be repeated in the training target
        # vocabulary, so the switch distribution receives full mass.
        switch_log_prob = torch.where(has_repeat_class, switch_log_prob, torch.zeros_like(switch_log_prob))
        result = switch_log_probs + switch_log_prob[:, None]
        return torch.where(mask, repeat_log_prob[:, None], result)


In [2]:
cfg=dict(embedding_dim=32,hidden_dim=64,max_context=32,dropout=.2,repeat_class=torch.tensor([-1,-1,0,1,2,3,4,0,1,2]))
model=UNCP6NextClick(10,5,cfg)
x=torch.tensor([[2,2,3,0],[4,5,5,7]])
lengths=torch.tensor([3,4])
out=model(x,lengths)
assert out.shape==(2,5) and torch.allclose(out.exp().sum(1),torch.ones(2),atol=1e-5)
print('UNCP 6 forward pass OK:',tuple(out.shape),'normalized probabilities')

UNCP 6 forward pass OK: (2, 5) normalized probabilities


## Recorded experiment results

The following cell executes a consistency check over the recorded three-seed result rows and displays them. It does not retrain the model.

In [3]:
import json, numpy as np
record = json.loads("{\"summary\": {\"protocol\": \"grouped_user\", \"model\": \"uncp6\", \"seeds\": \"3\", \"test_examples\": \"10833\", \"test_users\": \"11\", \"top1\": \"0.6659897227607002\", \"top3\": \"0.840825871565279\", \"macro_target_top1\": \"0.5702779086941139\", \"macro_target_top3\": \"0.7281072101547532\", \"macro_user_top1\": \"0.6656282403262328\", \"macro_user_top3\": \"0.8418568215426565\", \"cluster_ci95_top1_low\": \"0.6091920450994863\", \"cluster_ci95_top1_high\": \"0.6995171213041147\", \"cluster_ci95_top3_low\": \"0.802033844632611\", \"cluster_ci95_top3_high\": \"0.8642404745892035\", \"top1_seed_sd\": \"0.005097458602388681\", \"top3_seed_sd\": \"0.0021635614426593584\", \"macro_target_top1_seed_sd\": \"0.011683224489405347\", \"macro_target_top3_seed_sd\": \"0.011689404952768491\", \"macro_user_top1_seed_sd\": \"0.005838459964558864\", \"macro_user_top3_seed_sd\": \"0.002589649228566629\"}, \"runs\": [{\"protocol\": \"grouped_user\", \"model\": \"uncp6\", \"seed\": \"11\", \"test_examples\": \"10833\", \"test_users\": \"11\", \"top1\": \"0.6703590879719376\", \"top3\": \"0.838364257361765\", \"macro_target_top1\": \"0.5756091800875947\", \"macro_target_top3\": \"0.7365269857757945\", \"macro_user_top1\": \"0.6678463424106305\", \"macro_user_top3\": \"0.8396346576501166\", \"best_epoch\": \"24\", \"epochs_run\": \"29\"}, {\"protocol\": \"grouped_user\", \"model\": \"uncp6\", \"seed\": \"23\", \"test_examples\": \"10833\", \"test_users\": \"11\", \"top1\": \"0.6603895504477061\", \"top3\": \"0.842425920797563\", \"macro_target_top1\": \"0.5568800368153548\", \"macro_target_top3\": \"0.7147609816096553\", \"macro_user_top1\": \"0.6590057845179073\", \"macro_user_top3\": \"0.8447007548538839\", \"best_epoch\": \"20\", \"epochs_run\": \"25\"}, {\"protocol\": \"grouped_user\", \"model\": \"uncp6\", \"seed\": \"37\", \"test_examples\": \"10833\", \"test_users\": \"11\", \"top1\": \"0.6672205298624573\", \"top3\": \"0.8416874365365088\", \"macro_target_top1\": \"0.5783445091793922\", \"macro_target_top3\": \"0.7330336630788102\", \"macro_user_top1\": \"0.6700325940501609\", \"macro_user_top3\": \"0.8412350521239692\", \"best_epoch\": \"25\", \"epochs_run\": \"30\"}], \"behavior\": [{\"group\": \"all\", \"n\": 10833, \"uncp6_top1\": 0.6659897227607003, \"uncp6_top3\": 0.8408258715652789, \"uncp5_top1\": 0.6670974491522816, \"uncp5_top3\": 0.8508569494445982}, {\"group\": \"repeat\", \"n\": 1541, \"uncp6_top1\": 0.8407960199004977, \"uncp6_top3\": 0.9409474367293965, \"uncp5_top1\": 0.8654553320354749, \"uncp5_top3\": 0.9407311269738265}, {\"group\": \"switch\", \"n\": 9292, \"uncp6_top1\": 0.6369995695221696, \"uncp6_top3\": 0.8242215525900415, \"uncp5_top1\": 0.6342014636246233, \"uncp5_top3\": 0.8359520734682163}]}")
summary, runs = record['summary'], record['runs']
top1 = float(summary.get('test_mean_top1', summary.get('top1')))
top3 = float(summary.get('test_mean_top3', summary.get('top3')))
seed_top1 = np.array([float(row['top1']) for row in runs])
seed_top3 = np.array([float(row['top3']) for row in runs])
assert np.isclose(seed_top1.mean(), top1, atol=1e-9)
assert np.isclose(seed_top3.mean(), top3, atol=1e-9)
print('Recorded grouped-user experiment: 10,833 test targets, 11 held-out users')
print(f'UNCP 6  Top-1: {100*top1:.2f}%  Top-3: {100*top3:.2f}%')
print('Seeds 11, 23, 37 Top-3:', [round(100*x,2) for x in seed_top3])
if record['behavior']:
    for group in record['behavior']:
        print(group['group'], 'n=',group['n'],'UNCP6 Top-3=',round(100*group['uncp6_top3'],2),'UNCP5 Top-3=',round(100*group['uncp5_top3'],2))

Recorded grouped-user experiment: 10,833 test targets, 11 held-out users
UNCP 6  Top-1: 66.60%  Top-3: 84.08%
Seeds 11, 23, 37 Top-3: [np.float64(83.84), np.float64(84.24), np.float64(84.17)]
all n= 10833 UNCP6 Top-3= 84.08 UNCP5 Top-3= 85.09
repeat n= 1541 UNCP6 Top-3= 94.09 UNCP5 Top-3= 94.07
switch n= 9292 UNCP6 Top-3= 82.42 UNCP5 Top-3= 83.6
